In [2]:
%pip install boto3 -q

Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
import boto3

MINIO_ENDPOINT = "http://minio:9000"
MINIO_ACCESS_KEY = "minioadmin"
MINIO_SECRET_KEY = "minioadmin"

s3_client = boto3.client(
    "s3",
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    region_name="us-east-1",
)

In [2]:
response = s3_client.get_object(
    Bucket="raw",
    Key="BankChurners.csv",
)

bank_churners_raw = pd.read_csv(response["Body"])

print("Linhas:", bank_churners_raw.shape[0])
print("Colunas:", bank_churners_raw.shape[1])
print("Tamanho no MinIO:", response["ContentLength"], "bytes")
print("ETag:", response["ETag"])

display(bank_churners_raw.head())

Linhas: 10127
Colunas: 23
Tamanho no MinIO: 1510880 bytes
ETag: "c8c0eb266d15fba793ede87f94eddad8"


,CLIENTNUM,Attrition_Flag,Customer_Age,Gender,Dependent_count,Education_Level,Marital_Status,Income_Category,Card_Category,Months_on_book,...,Credit_Limit,Total_Revolving_Bal,Avg_Open_To_Buy,Total_Amt_Chng_Q4_Q1,Total_Trans_Amt,Total_Trans_Ct,Total_Ct_Chng_Q4_Q1,Avg_Utilization_Ratio,Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_1,Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_2
0,768805383,Existing Customer,45,M,3,High School,Married,$60K - $80K,Blue,39,...,12691.0,777,11914.0,1.335,1144,42,1.625,0.061,0.000093,0.99991
1,818770008,Existing Customer,49,F,5,Graduate,Single,Less than $40K,Blue,44,...,8256.0,864,7392.0,1.541,1291,33,3.714,0.105,0.000057,0.99994
2,713982108,Existing Customer,51,M,3,Graduate,Married,$80K - $120K,Blue,36,...,3418.0,0,3418.0,2.594,1887,20,2.333,0.000,0.000021,0.99998
3,769911858,Existing Customer,40,F,4,High School,Unknown,Less than $40K,Blue,34,...,3313.0,2517,796.0,1.405,1171,20,2.333,0.760,0.000134,0.99987
4,709106358,Existing Customer,40,M,3,Uneducated,Married,$60K - $80K,Blue,21,...,4716.0,0,4716.0,2.175,816,28,2.500,0.000,0.000022,0.99998


In [3]:
expected_target_values = {
    "Existing Customer",
    "Attrited Customer",
}

ingestion_validation = {
    "rows": len(bank_churners_raw),
    "columns": bank_churners_raw.shape[1],
    "unique_customers": bank_churners_raw["CLIENTNUM"].nunique(),
    "duplicate_customers": bank_churners_raw["CLIENTNUM"].duplicated().sum(),
    "total_nulls": int(bank_churners_raw.isna().sum().sum()),
    "target_values": set(
        bank_churners_raw["Attrition_Flag"].unique()
    ),
}

display(
    pd.Series(
        {
            **ingestion_validation,
            "target_values": sorted(
                ingestion_validation["target_values"]
            ),
        },
        name="value",
    ).to_frame()
)

assert ingestion_validation["rows"] == 10_127
assert ingestion_validation["columns"] == 23
assert ingestion_validation["unique_customers"] == 10_127
assert ingestion_validation["duplicate_customers"] == 0
assert ingestion_validation["total_nulls"] == 0
assert (
    ingestion_validation["target_values"]
    == expected_target_values
)

print("MinIO raw data validation passed.")

,value
rows,10127
columns,23
unique_customers,10127
duplicate_customers,0
total_nulls,0
target_values,"[Attrited Customer, Existing Customer]"


MinIO raw data validation passed.


In [4]:
from sqlalchemy import create_engine

DATABASE_URL = (
    "postgresql+psycopg2://"
    "finpulse_user:finpulse_password"
    "@postgres:5432/finpulse"
)

engine = create_engine(DATABASE_URL)

In [5]:
database_validation_query = """
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT client_num) AS unique_customers,
    COUNT(*) FILTER (
        WHERE client_num IS NULL
    ) AS null_customer_ids,
    COUNT(*) FILTER (
        WHERE attrition_flag = 'Existing Customer'
    ) AS existing_customers,
    COUNT(*) FILTER (
        WHERE attrition_flag = 'Attrited Customer'
    ) AS attrited_customers
FROM raw.bank_churners
"""

database_validation = pd.read_sql(
    database_validation_query,
    engine,
).iloc[0]

minio_target_counts = (
    bank_churners_raw["Attrition_Flag"]
    .value_counts()
)

comparison = pd.DataFrame(
    {
        "minio": {
            "rows": len(bank_churners_raw),
            "unique_customers": (
                bank_churners_raw["CLIENTNUM"].nunique()
            ),
            "existing_customers": int(
                minio_target_counts["Existing Customer"]
            ),
            "attrited_customers": int(
                minio_target_counts["Attrited Customer"]
            ),
        },
        "postgresql": {
            "rows": int(database_validation["rows"]),
            "unique_customers": int(
                database_validation["unique_customers"]
            ),
            "existing_customers": int(
                database_validation["existing_customers"]
            ),
            "attrited_customers": int(
                database_validation["attrited_customers"]
            ),
        },
    }
)

comparison["matches"] = (
    comparison["minio"]
    == comparison["postgresql"]
)

display(comparison)

assert database_validation["null_customer_ids"] == 0
assert comparison["matches"].all()

print("MinIO and PostgreSQL raw data validation passed.")

,minio,postgresql,matches
rows,10127,10127,True
unique_customers,10127,10127,True
existing_customers,8500,8500,True
attrited_customers,1627,1627,True


MinIO and PostgreSQL raw data validation passed.


In [6]:
pipeline_validation_query = """
SELECT
    'raw.bank_churners' AS dataset,
    COUNT(*) AS rows,
    COUNT(DISTINCT client_num) AS unique_customers,
    COUNT(*) FILTER (
        WHERE attrition_flag = 'Attrited Customer'
    ) AS churn_customers
FROM raw.bank_churners

UNION ALL

SELECT
    'staging.stg_bank_churners' AS dataset,
    COUNT(*) AS rows,
    COUNT(DISTINCT customer_id) AS unique_customers,
    COUNT(*) FILTER (
        WHERE churn_flag = 1
    ) AS churn_customers
FROM staging.stg_bank_churners

UNION ALL

SELECT
    'marts.mart_customer_churn_model' AS dataset,
    COUNT(*) AS rows,
    COUNT(DISTINCT customer_id) AS unique_customers,
    COUNT(*) FILTER (
        WHERE churn_flag = 1
    ) AS churn_customers
FROM marts.mart_customer_churn_model
"""

pipeline_validation = pd.read_sql(
    pipeline_validation_query,
    engine,
)

display(pipeline_validation)

assert pipeline_validation["rows"].eq(10_127).all()
assert pipeline_validation["unique_customers"].eq(10_127).all()
assert pipeline_validation["churn_customers"].eq(1_627).all()

print("MinIO → raw → staging → marts validation passed.")

,dataset,rows,unique_customers,churn_customers
0,marts.mart_customer_churn_model,10127,10127,1627
1,raw.bank_churners,10127,10127,1627
2,staging.stg_bank_churners,10127,10127,1627


MinIO → raw → staging → marts validation passed.


In [7]:
model_columns_query = """
SELECT column_name
FROM information_schema.columns
WHERE table_schema = 'marts'
  AND table_name = 'mart_customer_churn_model'
ORDER BY ordinal_position
"""

model_columns = (
    pd.read_sql(model_columns_query, engine)
    ["column_name"]
    .tolist()
)

forbidden_columns = [
    column
    for column in model_columns
    if (
        "naive_bayes" in column.lower()
        or column.lower() == "attrition_status"
    )
]

print("Total de colunas no mart:", len(model_columns))
print("Colunas proibidas encontradas:", forbidden_columns)

assert len(model_columns) == 21
assert forbidden_columns == []

print("Model mart leakage validation passed.")

Total de colunas no mart: 21
Colunas proibidas encontradas: []
Model mart leakage validation passed.
